In [3]:
import numpy as np
import pandas as pd
import os 
import glob
import chardet
from datetime import datetime

In [4]:
os.getcwd()

'c:\\Users\\Prithwijit\\Dropbox\\My PC (LAPTOP-7SET5QFO)\\Documents\\jupyternotebook_ws\\Database_code'

In [377]:
os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir))

# function for missing data

In [6]:
def find_missing_list_indices(dataframe, column_name, value_list):

    nan_ind = dataframe[dataframe[column_name] == 'nan'].index

    dataframe.drop(nan_ind,inplace=True)

    int_df_sample_id = [int(str(x)[-4:]) for x in dataframe[column_name].tolist()]

    int_value_list = [int(str(x)[-4:]) for x in value_list]

    indices = [index for index, value in enumerate(int_value_list) if value not in int_df_sample_id]

    dic_err = {str(i) : ['error'] for i in dataframe.columns}
    df_err = pd.DataFrame(dic_err)

    
    for index in indices:
        dataframe = pd.concat([dataframe.iloc[:index], df_err, dataframe.iloc[index:]]).reset_index(drop=True)
    result_df = dataframe.reset_index(drop=True)

    result_df.iloc[nan_ind,:] = 'nan'
    
    return result_df

# KT20 cond and mag

In [220]:
def cond_sus_proc(drillhole,prop):

    data_inventory = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name = None)
    df_record = data_inventory.get(prop)
    df_report = data_inventory.get('sample_report') 
    sample_id_report = df_report['sample_id'].to_list()

    os.chdir(f'{drillhole}/cond_sus')
    

    dic_result = {'sample id':[],'mean':[], 'std':[],'median':[], 'max':[],'min':[]} ## creating dataframe for results #######


    date_objects = [datetime.strptime(date_str, "%d-%m-%y") for date_str in os.listdir()]

    # Sort the list based on datetime objects
    sorted_folder_names = [date.strftime("%d-%m-%y") for date in sorted(date_objects)]

    
    for dir in sorted_folder_names:
        os.chdir(dir+'/'+'kt20_0385')

        try:
            meta_data = pd.read_csv('kappa_sigma_0385.csv')
            df_record_sub = df_record[df_record['measurement_id'].isin(meta_data['Record_Id'])]
            
        except KeyError:
            meta_data = pd.read_csv('kappa_sigma_0385.csv',sep=';')
            df_record_sub = df_record[df_record['measurement_id'].isin(meta_data['Record_Id'])]

                                  
        record_id_list_sub = df_record_sub['measurement_id'].tolist()
        record_id_strings_sub = [str(int(record_id)) for record_id in record_id_list_sub] 
        sample_id_sub = df_record_sub['sample_id'].tolist()
        sample_id_strings_sub = [str(sample_id) for sample_id in sample_id_sub]

        for i in range(len(record_id_strings_sub)):
            os.chdir(str(record_id_strings_sub[i]))   #### changing directory iteratively##########
            file = glob.glob('*.csv')
            df = pd.read_csv(file[0])    
            a1 = df.index[df['Date'] == 'Min'].to_list()
            a1 = int(np.array(a1))
            d1 = np.array(df.iloc[2:a1,1])
            d1 = d1.astype('float32')
            d1 = np.reshape(d1, (a1-2,1))
            w1 = np.ma.masked_outside(d1,np.mean(d1)+(2*np.std(d1)),np.mean(d1)-(2*np.std(d1)))
            dic_result['sample id'].append(sample_id_strings_sub[i]),dic_result['mean'].append(np.mean(w1)),dic_result['std'].append(np.std(w1)),dic_result['median'].append(np.median(w1)),dic_result['max'].append(np.max(w1)),dic_result['min'].append(np.min(w1))
            os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir))
            # print(os.getcwd())
        os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir+os.sep + os.pardir))

    os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir + os.sep + os.pardir))

    df_proc = pd.DataFrame(dic_result)

    nan_indices = df_record[df_record['measurement_id'].isna()].index

    dic_nan = {str(i) : ['nan'] for i in df_proc.columns}
    df_nan = pd.DataFrame(dic_nan)

    
    for index in nan_indices:
        df_proc = pd.concat([df_proc.iloc[:index], df_nan, df_proc.iloc[index:]]).reset_index(drop=True)
    result_df = df_proc.reset_index(drop=True)

    # result_df.iloc[:,0] = (sample_id).astype(str)
    cal = result_df['sample id'].str.contains('cal')
    df_cal = result_df[cal]
    df_final = result_df[np.logical_not(cal)]
    df_final.index = range(len(df_final))
    df_final = find_missing_list_indices(df_final,'sample id',sample_id_report)

    return df_final,df_cal

In [221]:
d1,d2 = cond_sus_proc('TC-2531-001','sus')

c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-p

In [223]:
d1.to_csv('check.csv')

# Gamma processing

In [48]:
def gamma(drillhole):     
    
    data_inventory = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name = None)
    df_record = data_inventory.get('gamma')
    df_report = data_inventory.get('sample_report') 
    sample_id_report = df_report['sample_id'].to_list()
    
    os.chdir(f'{drillhole}/gamma')

    # measurement_id_array = np.c_[df_record['measurement_id1'],df_record['measurement_id2'],df_record['measurement_id3']]
    dic_result = {'sample id':[],'K_mean':[],'K_median':[], 'K_max':[],'K_min':[], 'K_std':[],'U_mean':[],'U_median':[], 
                  'U_max':[],'U_min':[], 'U_std':[],'Th_mean':[],'Th_median':[], 'Th_max':[],'Th_min':[], 'Th_std':[]}
    
    dic_cal = {'sample id':[],'K_cal':[],'U_cal':[],'Th_cal':[]}

    date_objects = [datetime.strptime(date_str, "%d-%m-%y") for date_str in os.listdir()]

    # Sort the list based on datetime objects
    sorted_folder_names = [date.strftime("%d-%m-%y") for date in sorted(date_objects)]

    
    for dir in sorted_folder_names:
        os.chdir(f'{dir}/gt40_0154/assay')

        # file_record = glob.glob('*.xlsx')
        # df_record = pd.read_excel(file_record[0])
        try:
            file_assay = glob.glob('*.csv')
            assay = pd.read_csv(file_assay[0],sep=';')
            assay.columns = assay.columns.str.strip()
            df_assay = assay[assay['Id'].notna()]
        except KeyError:
            try:
                file_assay = glob.glob('*.csv')
                assay = pd.read_csv(file_assay[0],sep=',')
                assay.columns = assay.columns.str.strip()
                df_assay = assay[assay['Id'].notna()]
            except KeyError:
                print('cannot read the .csv file')
        
        df_record_sub = df_record[df_record['measurement_id1'].isin(df_assay['Id']) + 
                                  df_record['measurement_id2'].isin(df_assay['Id']) + df_record['measurement_id3'].isin(df_assay['Id'])]
        sampleid = np.array(df_record_sub['sample_id'])
        measurement_id_array = np.c_[df_record_sub['measurement_id1'],df_record_sub['measurement_id2'],df_record_sub['measurement_id3']] 

        
        for i in range (len(df_record_sub)):
            measurement_id_temp = measurement_id_array[i,:].tolist()
            df_temp = df_assay[df_assay['Id'].isin(measurement_id_temp)]
            df_temp.replace('<  0.00000', '0', inplace=True)


            if str(sampleid[i]).startswith('cal'):
                
                dic_cal['sample id'].append(str(sampleid[i])),dic_cal['K_cal'].append(float(df_temp['Component_1 value'].to_list()[0])),dic_cal['U_cal'].append(float(df_temp['Component_2 value'].to_list()[1])),
                dic_cal['Th_cal'].append(float(df_temp['Component_3 value'].to_list()[2]))

            else:
                dic_result['sample id'].append(str(sampleid[i])),dic_result['K_mean'].append(df_temp['Component_1 value'].astype('float16').mean()),
                dic_result['K_median'].append(df_temp['Component_1 value'].astype('float16').median()),dic_result['K_max'].append(df_temp['Component_1 value'].astype('float16').max()),
                dic_result['K_min'].append(df_temp['Component_1 value'].astype('float16').min()),dic_result['K_std'].append(df_temp['Component_1 value'].astype('float16').std()),
                dic_result['U_mean'].append(df_temp['Component_2 value'].astype('float16').mean()),dic_result['U_median'].append(df_temp['Component_2 value'].astype('float16').median()),
                dic_result['U_max'].append(df_temp['Component_2 value'].astype('float16').max()),dic_result['U_min'].append(df_temp['Component_2 value'].astype('float16').min()),
                dic_result['U_std'].append(df_temp['Component_2 value'].astype('float16').std()),
                dic_result['Th_mean'].append(df_temp['Component_3 value'].astype('float16').mean()),dic_result['Th_median'].append(df_temp['Component_3 value'].astype('float16').median()),
                dic_result['Th_max'].append(df_temp['Component_3 value'].astype('float16').max()),dic_result['Th_min'].append(df_temp['Component_3 value'].astype('float16').min()),dic_result['Th_std'].append(df_temp['Component_3 value'].astype('float16').std())

        

        os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir + os.sep + os.pardir + os.sep + os.pardir))   

    os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir+os.sep + os.pardir))

    df_proc = pd.DataFrame(dic_result)
    df_cal = pd.DataFrame(dic_cal)
    # sample_id = df_proc['sample id']

    nan_indices = df_record[df_record['measurement_id1'].isna()].index
    dic_nan = {str(i) : ['nan'] for i in df_proc.columns}
    df_nan = pd.DataFrame(dic_nan)

    for index in nan_indices:
        df_proc = pd.concat([df_proc.iloc[:index], df_nan, df_proc.iloc[index:]]).reset_index(drop=True)
    result_df = df_proc.reset_index(drop=True)

    # result_df.iloc[:,0] = (sample_id).astype(str)

    rem = result_df['sample id'].str.contains('cal')
    # df_rem = result_df[rem]
    df_final = result_df[np.logical_not(rem)]
    df_final.index = range(len(df_final))
    df_final = find_missing_list_indices(df_final,'sample id',sample_id_report)

    return df_final,df_cal

In [49]:
a,b = gamma('TC-2638-088')

C:\Users\Prithwijit\AppData\Local\Temp\ipykernel_13492\3459024718.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp.replace('<  0.00000', '0', inplace=True)
C:\Users\Prithwijit\AppData\Local\Temp\ipykernel_13492\3459024718.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp.replace('<  0.00000', '0', inplace=True)
C:\Users\Prithwijit\AppData\Local\Temp\ipykernel_13492\3459024718.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-

In [45]:
os.getcwd()

'c:\\Users\\Prithwijit\\Dropbox\\My PC (LAPTOP-7SET5QFO)\\Documents\\jupyternotebook_ws\\Database_code'

In [44]:
os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir))

# Density

In [167]:
def density(drillhole):
    
    dict = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name = None)
    df_den = dict.get('den') 
    df_out = pd.concat([df_den['sample_id'],df_den['density']],axis=1)
    df_out.rename(columns={'sample_id': 'sample id'}, inplace=True)

    return df_out

# IP

In [8]:
def ip_res (drillhole):

    data_inventory = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name = None)
    df_record = data_inventory.get('ip_res')
    sample_id = df_record['sample_id']
    df_record_nan_rem = df_record[~df_record['measurement_id'].isna()]

    os.chdir(f'{drillhole}/ip_res')

    df_proc = pd.DataFrame()

    date_objects = [datetime.strptime(date_str, "%d-%m-%y") for date_str in os.listdir()]

    # Sort the list based on datetime objects
    sorted_folder_names = [date.strftime("%d-%m-%y") for date in sorted(date_objects)]
    
    for dir in sorted_folder_names:

        os.chdir(dir+'/'+'kt20_0385'+'/'+'ip')
        ip_record = pd.read_csv('ip_records.csv')
        for i in df_record_nan_rem['measurement_id']:
            df_proc_temp = ip_record[ip_record.Record_Id==i]
            df_proc = pd.concat([df_proc, df_proc_temp], ignore_index=True)

        os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir+os.sep + os.pardir+os.sep + os.pardir))

    os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir + os.sep + os.pardir))

    nan_indices = df_record[df_record['measurement_id'].isna()].index

    dic_nan = {str(i) : ['nan'] for i in df_proc.columns}
    df_nan = pd.DataFrame(dic_nan)

    
    for index in nan_indices:
        df_proc = pd.concat([df_proc.iloc[:index], df_nan, df_proc.iloc[index:]]).reset_index(drop=True)
    result_df = df_proc.reset_index(drop=True)

    result_df[result_df['Record_Id']=='nan']=-1
    proc_meas_id_with_nan = result_df['Record_Id'].to_list()
    df_record[df_record['measurement_id'].isna()]=-1 
    df_rec_meas_id = df_record['measurement_id'].to_list()
    ind_bool_err = []
    for i in df_rec_meas_id:       # df_test has to be the df_record
        ind_bool_err.append(i not in proc_meas_id_with_nan)
    ind_err=[]
    for i in range(len(ind_bool_err)):
        if ind_bool_err[i]==True:
            ind_err.append(i)
    df_processed_data_with_nan_rem = result_df[result_df['Record_Id']!=-1]

    a = result_df['Record_Id']==-1
    nan_ind=[]
    for i in range(len(a)):
        if a[i]==True:
            nan_ind.append(i)

    nan_ind.extend(ind_err)
    nan_ind.sort()

    dic_nan = {str(i) : ['nan'] for i in df_processed_data_with_nan_rem.columns}
    df_nan = pd.DataFrame(dic_nan) 
    for index in nan_ind:
        df_processed_data_with_nan_rem = pd.concat([df_processed_data_with_nan_rem.iloc[:index], df_nan,df_processed_data_with_nan_rem.iloc[index:]]).reset_index(drop=True)
    result_df = df_processed_data_with_nan_rem.reset_index(drop=True)

    result_df.iloc[ind_err,:] = 'error'

    result_df.iloc[:,0] = (sample_id).astype(str)
    result_df.rename(columns={'Record_Id': 'sample id'}, inplace=True)
    cal = result_df['sample id'].str.contains('cal')
    df_cal = result_df[cal]
    df_final = result_df[np.logical_not(cal)]
    df_final.index = range(len(df_final))

    return df_final,df_cal

In [9]:
a,b=ip_res('campaign02')

In [95]:
a.to_csv('check.csv')

In [531]:
record,result = ip_res('campaign02')

In [443]:
result.to_csv('check.csv')

# Velocity

In [21]:
def convert_csv_to_utf8(input_file):
    with open(input_file, 'rb') as f:
        enc = chardet.detect(f.read())

    if enc['encoding'].lower() == 'utf-8':
        print(f"The file {input_file} is already in UTF-8 encoding.")
    else:
        # Use a temporary DataFrame to read the CSV with the detected encoding
        df = pd.read_csv(input_file, encoding=enc['encoding'])
        
        # Save the DataFrame with UTF-8 encoding
        output_file = input_file.replace('.csv', '_utf8.csv')
        df.to_csv(output_file, encoding='utf-8', sep=',',index=False)
        print(f"File converted and saved as {output_file}")

In [96]:
def vel(drillhole,prop):

    data_inventory = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name = None)
    df_record = data_inventory.get(prop)
    sample_id = df_record['sample_id']
    df_record_nan_rem = df_record[~df_record['measurement_id'].isna()]

    os.chdir(f'{drillhole}/{prop}')

    df_proc = pd.DataFrame()

    date_objects = [datetime.strptime(date_str, "%d-%m-%y") for date_str in os.listdir()]

    # Sort the list based on datetime objects
    sorted_folder_names = [date.strftime("%d-%m-%y") for date in sorted(date_objects)]

    for dir in sorted_folder_names:
        os.chdir(dir)
        
        convert_csv_to_utf8('Punditlink Data.csv')

        file = glob.glob('*.csv')

        rawdata = open(file[len(file)-1], 'rb').read()
        enc = chardet.detect(rawdata)

        try:
            df_data = pd.read_csv(file[len(file)-1],sep=',', encoding = enc['encoding'],header = 14)
            df_data.columns = df_data.columns.str.strip()
            record_id = df_data['ID']
            a=[]
            for i in range(1,len(df_data),25):
                a.append(i)

        except KeyError:

            try:
                df_data = pd.read_csv(file[len(file)-1],sep='\t', encoding = enc['encoding'],header = 10)
                df_data.columns = df_data.columns.str.strip()
                record_id = df_data['ID']
                a=[]
                for i in range(1,len(df_data),16):
                    a.append(i)
            except KeyError:
                
                try:
                    df_data = pd.read_csv(file[len(file)-1],sep=';', encoding = enc['encoding'],header = 14)
                    df_data.columns = df_data.columns.str.strip()
                    record_id = df_data['ID']
                    a=[]
                    for i in range(1,len(df_data),25):
                        a.append(i)

                except KeyError:
                    print('none of the attempts works')
  
        df_vel = df_data.iloc[a]
        df_proc = pd.concat([df_proc, df_vel], ignore_index=True)
        
        os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir))

    df_proc_comp = pd.DataFrame()

    for i in df_record_nan_rem['measurement_id']:
        df_proc_temp = df_proc[df_proc.ID.str.strip()==str(int(i))]
        df_proc_comp = pd.concat([df_proc_comp, df_proc_temp], ignore_index=True)

        

    os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir + os.sep + os.pardir))

    nan_indices = df_record[df_record['measurement_id'].isna()].index

    dic_nan = {str(i) : ['nan'] for i in df_proc_comp.columns}
    df_nan = pd.DataFrame(dic_nan)

    
    for index in nan_indices:
        df_proc_comp = pd.concat([df_proc_comp.iloc[:index], df_nan, df_proc_comp.iloc[index:]]).reset_index(drop=True)
    result_df = df_proc_comp.reset_index(drop=True)

    result_df[result_df['ID']=='nan']=-1
    proc_meas_id_with_nan = result_df['ID'].to_list()
    proc_meas_id_with_nan_int = [int(x) for x in proc_meas_id_with_nan]

    df_record[df_record['measurement_id'].isna()]=-1 
    df_rec_meas_id = df_record['measurement_id'].to_list()
    df_rec_meas_id_int = [int(x) for x in df_rec_meas_id]

    ind_bool_err = []
    for i in df_rec_meas_id_int:       
        ind_bool_err.append(i not in proc_meas_id_with_nan_int)
    ind_err=[]
    for i in range(len(ind_bool_err)):
        if ind_bool_err[i]==True:
            ind_err.append(i)
    df_processed_data_with_nan_rem = result_df[result_df['ID']!=-1]

    a = result_df['ID']==-1
    nan_ind=[]
    for i in range(len(a)):
        if a[i]==True:
            nan_ind.append(i)

    nan_ind.extend(ind_err)
    nan_ind.sort()

    dic_nan = {str(i) : ['nan'] for i in df_processed_data_with_nan_rem.columns}
    df_nan = pd.DataFrame(dic_nan) 
    for index in nan_ind:
        df_processed_data_with_nan_rem = pd.concat([df_processed_data_with_nan_rem.iloc[:index], df_nan,df_processed_data_with_nan_rem.iloc[index:]]).reset_index(drop=True)
    result_df = df_processed_data_with_nan_rem.reset_index(drop=True)

    result_df.iloc[ind_err,:] = 'error'

    result_df.rename(columns={'ID': 'sample id'}, inplace=True)

    result_df.iloc[:,0] = (sample_id).astype(str)
    result_df.index = range(len(result_df))

    return result_df

In [97]:
result_df = vel('campaign02','v_p')

File converted and saved as Punditlink Data_utf8.csv


In [98]:
result_df.to_csv('check.csv')

In [149]:
os.getcwd()

'c:\\Users\\Prithwijit\\Dropbox\\My PC (LAPTOP-7SET5QFO)\\Documents\\jupyternotebook_ws'

In [148]:
os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir))

# XRF

In [3]:
def extract_last_four_digits(number):
    return float(str(number)[-4:])

In [4]:
def xrf(drillhole):
        
        data_inventory = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name = None)
        df_record = data_inventory.get('sample_report')
        sample_id_report = df_record['sample_id'].to_list()

        
        os.chdir(f'{drillhole}/xrf')  

        
        df_proc = pd.DataFrame()

        date_objects = [datetime.strptime(date_str, "%d-%m-%y") for date_str in os.listdir()]

        # Sort the list based on datetime objects
        sorted_folder_names = [date.strftime("%d-%m-%y") for date in sorted(date_objects)]

        for dir in sorted_folder_names:
            os.chdir(dir)
            file = glob.glob('*.csv')
            try:
                df_xrf = pd.read_csv(file[len(file)-1],skiprows=1)
                info = df_xrf['info']
            except KeyError:
                try:
                    df_xrf = pd.read_csv(file[len(file)-1])
                    info = df_xrf['info']
                except KeyError:
                    try:
                        df_xrf = pd.read_csv(file[len(file)-1],sep=';')
                        info = df_xrf['info']
                    except KeyError:
                        try:
                            df_xrf = pd.read_csv(file[len(file)-1],sep='\t')
                            info = df_xrf['info']
                        except KeyError:
                            print('cannot read the .csv file')
                    
                
            os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir))
            df_proc = pd.concat([df_proc, df_xrf], ignore_index=True)

        info = df_proc['info'].tolist()
        
        i = 0
        drop_list = []

        while i < len(info):
            if isinstance(info[i], str) and info[i].startswith('cal'):
                drop_list.append(i)
            if isinstance(info[i], str) and info[i].startswith('std'):
                drop_list.append(i)
            if isinstance(info[i], str) and info[i].startswith('blk'):
                drop_list.append(i)
            i += 1
        # print(drop_list)
        new_df = df_proc.drop(df_proc.index[drop_list])

        # display(new_df)

        new_df.reset_index(inplace = True)
        new_df = new_df.drop(new_df[new_df['info'] == 'del'].index)
        new_info = new_df['info'].tolist()
        sampleid_list = []
        # delete the last letter of info
        for temp_info in new_info:
            sampleid = str(temp_info)[:-1]
            sampleid_list.append(sampleid)
        # delete NaN & abcdef columns
        new_df['info']=sampleid_list
        new_df.index = range(len(new_df))


        str_sampleid_report = [str(x) for x in sample_id_report]

        ind = []
        for i in range(len(sampleid_list)):
            if str(sampleid_list[i]) not in str_sampleid_report:
                ind.append(i)

        for i in sorted(ind,reverse=True):
            del sampleid_list[i]

        new_df = new_df.drop(new_df.index[ind])


        new_df = new_df.dropna(axis=1, how='all') #drop all rows that have any NaN values
        new_df.columns = new_df.columns.str.strip()
        new_df =new_df.drop( ['Instrument Serial Num','Reading #','Date','Time','Method Name','Test Label','Collimation Status','Units'],axis=1)
        # delete 'errors' columns
        new_df1=new_df.filter(regex='Error')
        new_df2=new_df.filter(regex='User')
        # new_df1=new_df.filter(regex='Compound')
        new_df = new_df.drop(new_df1,axis=1)
        new_df = new_df.drop(new_df2,axis=1)
        new_df = new_df.replace('<LOD','0.0')
        new_df = new_df.replace('na',np.NaN)
        new_df = new_df.replace('de',np.NaN)
        new_df1 = new_df.drop(['info'],axis=1)


        new_df = pd.concat([new_df1.apply(np.float64),new_df['info']],axis=1)
        new_df['temp_info'] = new_df['info'].apply(extract_last_four_digits)

        new_df1 = new_df.groupby(['temp_info'])[['Mg Concentration','Al Concentration','Si Concentration','P Concentration','S Concentration','K Concentration','Ca Concentration','Ti Concentration','V Concentration','Cr Concentration','Mn Concentration','Fe Concentration','Mg Concentration','Co Concentration','Ni Concentration','Cu Concentration','Zn Concentration','As Concentration','Se Concentration','Rb Concentration','Sr Concentration','Y Concentration','Zr Concentration','Nb Concentration','Mo Concentration','Ag Concentration','Cd Concentration','Sn Concentration','Sb Concentration','W Concentration','Au Concentration','Hg Concentration','Pb Concentration','Bi Concentration','Th Concentration','U Concentration','LE Concentration']].mean().reset_index()
        new_df2 = new_df.groupby(['temp_info'])[['Mg Concentration','Al Concentration','Si Concentration','P Concentration','S Concentration','K Concentration','Ca Concentration','Ti Concentration','V Concentration','Cr Concentration','Mn Concentration','Fe Concentration','Mg Concentration','Co Concentration','Ni Concentration','Cu Concentration','Zn Concentration','As Concentration','Se Concentration','Rb Concentration','Sr Concentration','Y Concentration','Zr Concentration','Nb Concentration','Mo Concentration','Ag Concentration','Cd Concentration','Sn Concentration','Sb Concentration','W Concentration','Au Concentration','Hg Concentration','Pb Concentration','Bi Concentration','Th Concentration','U Concentration','LE Concentration']].std().reset_index()
        new_df3 = new_df.groupby(['temp_info'])[['Mg Concentration','Al Concentration','Si Concentration','P Concentration','S Concentration','K Concentration','Ca Concentration','Ti Concentration','V Concentration','Cr Concentration','Mn Concentration','Fe Concentration','Mg Concentration','Co Concentration','Ni Concentration','Cu Concentration','Zn Concentration','As Concentration','Se Concentration','Rb Concentration','Sr Concentration','Y Concentration','Zr Concentration','Nb Concentration','Mo Concentration','Ag Concentration','Cd Concentration','Sn Concentration','Sb Concentration','W Concentration','Au Concentration','Hg Concentration','Pb Concentration','Bi Concentration','Th Concentration','U Concentration','LE Concentration']].median().reset_index()
        new_df4 = new_df.groupby(['temp_info'])[['Mg Concentration','Al Concentration','Si Concentration','P Concentration','S Concentration','K Concentration','Ca Concentration','Ti Concentration','V Concentration','Cr Concentration','Mn Concentration','Fe Concentration','Mg Concentration','Co Concentration','Ni Concentration','Cu Concentration','Zn Concentration','As Concentration','Se Concentration','Rb Concentration','Sr Concentration','Y Concentration','Zr Concentration','Nb Concentration','Mo Concentration','Ag Concentration','Cd Concentration','Sn Concentration','Sb Concentration','W Concentration','Au Concentration','Hg Concentration','Pb Concentration','Bi Concentration','Th Concentration','U Concentration','LE Concentration']].max().reset_index()
        new_df5 = new_df.groupby(['temp_info'])[['Mg Concentration','Al Concentration','Si Concentration','P Concentration','S Concentration','K Concentration','Ca Concentration','Ti Concentration','V Concentration','Cr Concentration','Mn Concentration','Fe Concentration','Mg Concentration','Co Concentration','Ni Concentration','Cu Concentration','Zn Concentration','As Concentration','Se Concentration','Rb Concentration','Sr Concentration','Y Concentration','Zr Concentration','Nb Concentration','Mo Concentration','Ag Concentration','Cd Concentration','Sn Concentration','Sb Concentration','W Concentration','Au Concentration','Hg Concentration','Pb Concentration','Bi Concentration','Th Concentration','U Concentration','LE Concentration']].min().reset_index()
        new_df1.columns = new_df1.columns.str.replace('Concentration', 'Mean')
        # new_df1.columns = new_df1.replace('Concentration', 'Mean')
        new_df2.columns = new_df2.columns.str.replace('Concentration', 'Std')
        new_df3.columns = new_df3.columns.str.replace('Concentration', 'Median')
        new_df4.columns = new_df4.columns.str.replace('Concentration', 'Max')
        new_df5.columns = new_df5.columns.str.replace('Concentration', 'Min')
        new_df6 = pd.concat([new_df1,new_df2,new_df3,new_df4,new_df5],axis=1)
        new_df7 = new_df6.sort_index(axis=1)
        new_df7 = new_df7.loc[:, ~new_df7.columns.duplicated()]
        new_df7 = new_df7.iloc[:, :-1]


        new_df7['info']= np.unique(sampleid_list)


        df_final = new_df7 
        
        df_final.rename(columns={'info': 'sample id'}, inplace=True)

        os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir+ os.sep + os.pardir))

        nan_ind = df_final[df_final['sample id'] == 'nan'].index

        df_final.drop(nan_ind,inplace=True)

        int_sample_id_report = [int(str(x)[-4:]) for x in df_final['sample id'].tolist()]

        int_df_sample_id = [int(str(x)[-4:]) for x in sample_id_report]


        indices = [index for index, value in enumerate(int_df_sample_id) if value not in int_sample_id_report]

        dic_err = {str(i) : ['error'] for i in df_final.columns}
        df_err = pd.DataFrame(dic_err)


        for index in indices:
            df_final = pd.concat([df_final.iloc[:index], df_err, df_final.iloc[index:]]).reset_index(drop=True)
        result_df = df_final.reset_index(drop=True)

        result_df.iloc[nan_ind,:] = 'nan'

        return result_df

In [6]:
df = xrf('campaign02')

In [7]:
df

,Ag Max,Ag Mean,Ag Median,Ag Min,Ag Std,Al Max,Al Mean,Al Median,Al Min,Al Std,...,Zn Mean,Zn Median,Zn Min,Zn Std,Zr Max,Zr Mean,Zr Median,Zr Min,Zr Std,sample id
0,0.0,0.0,0.0,0.0,0.0,3503.0,1894.857143,2831.0,0.0,1787.284572,...,13960.428571,9933.0,255.0,15448.81306,12.0,3.285714,0.0,0.0,5.618846,A0057
1,165.0,51.333333,0.0,0.0,79.828984,8881.0,6054.833333,5897.5,3953.0,1618.363669,...,52532.0,32303.0,4012.0,61069.205844,19.0,3.166667,0.0,0.0,7.756718,A0058
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1203.333333,1136.0,345.0,735.518774,0.0,0.0,0.0,0.0,0.0,A0059
3,0.0,0.0,0.0,0.0,0.0,4963.0,3498.0,3590.5,1996.0,1219.651262,...,58391.0,57073.0,33568.0,22856.578108,0.0,0.0,0.0,0.0,0.0,A0060
4,0.0,0.0,0.0,0.0,0.0,23693.0,11809.333333,11537.0,741.0,10990.245069,...,8097.5,4936.5,371.0,8847.58215,83.0,37.0,38.0,0.0,36.562276,A0061
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,0.0,0.0,0.0,0.0,0.0,8805.0,6742.333333,6681.5,5009.0,1570.239939,...,215130.333333,225561.0,69968.0,88015.781032,0.0,0.0,0.0,0.0,0.0,A0187
131,0.0,0.0,0.0,0.0,0.0,10243.0,6447.5,5995.0,2965.0,2537.487478,...,309668.5,316654.0,196174.0,63336.399856,0.0,0.0,0.0,0.0,0.0,A0188
132,0.0,0.0,0.0,0.0,0.0,18738.0,12399.833333,12047.0,7146.0,4827.847405,...,380867.333333,391043.0,325774.0,46904.293252,0.0,0.0,0.0,0.0,0.0,A0189
133,0.0,0.0,0.0,0.0,0.0,10188.0,5057.0,4771.0,1891.0,3168.713872,...,83117.166667,71702.5,2205.0,70755.791173,0.0,0.0,0.0,0.0,0.0,A0190


In [206]:
df.to_csv('check.csv')

In [73]:
os.getcwd()

'c:\\Users\\Prithwijit\\Dropbox\\My PC (LAPTOP-7SET5QFO)\\Documents\\jupyternotebook_ws\\Database_code'

In [72]:
os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir+ os.sep + os.pardir))
